# EV Adoption Behavior - 02. Data Cleaning

Applies the cleaning rules established in `01_data_understanding.ipynb`. Each step states the *why*, not just the *what* - useful for explaining decisions later (interviews, portfolio writeups).

## 1. Load data

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('global_ev_adoption_behavior_2026.csv')
df.shape

## 2. Fix `fuel_expense_per_month` negative values

Negative monthly expense isn't physically possible - treated as a sign/entry error rather than a true negative value, so we take the absolute value rather than dropping these rows (dropping would lose otherwise-valid data for no good reason).

In [ ]:
neg_count = (df['fuel_expense_per_month'] < 0).sum()
print(f'Negative values before fix: {neg_count}')

df['fuel_expense_per_month'] = df['fuel_expense_per_month'].abs()

neg_count_after = (df['fuel_expense_per_month'] < 0).sum()
print(f'Negative values after fix: {neg_count_after}')

## 3. Review `vehicle_age_years == 0`

Kept as-is: a vehicle age of 0 plausibly represents a brand-new vehicle purchased this year, not a data error. Re-check after the fuel fix that these rows don't look suspicious in other columns before moving on - if something else looks broken for these same rows, revisit this decision.

In [ ]:
df[df['vehicle_age_years'] == 0].describe()

## 4. Impute `education_level` (categorical → mode)

In [ ]:
mode_val = df['education_level'].mode()[0]
print(f'Mode: {mode_val}')

df['education_level'] = df['education_level'].fillna(mode_val)

## 5. Impute `charging_station_accessibility` (median **per `city_type`**)

A global median would blur Urban (~7) and Rural/Suburban (~5) into one wrong number for both groups, so we fill each row using its own city_type's median instead.

In [ ]:
df['charging_station_accessibility'] = (
    df.groupby('city_type')['charging_station_accessibility']
      .transform(lambda x: x.fillna(x.median()))
)

## 6. Impute `ev_knowledge_score` (overall median)

No meaningful difference across `city_type` groups here, so a single overall median is sufficient - no need for the extra complexity of group-wise imputation.

In [ ]:
median_val = df['ev_knowledge_score'].median()
print(f'Median: {median_val}')

df['ev_knowledge_score'] = df['ev_knowledge_score'].fillna(median_val)

## 7. Verify - no nulls remain, no negative fuel expense, shape unchanged

In [ ]:
print('Remaining nulls:\n', df.isnull().sum()[df.isnull().sum() > 0])
print('\nNegative fuel expense rows:', (df['fuel_expense_per_month'] < 0).sum())
print('\nFinal shape:', df.shape)

## 8. Save cleaned dataset

This cleaned file is what `03_eda.ipynb` (and eventually `app.py`) should read from - keep the raw CSV untouched as your source of truth.

In [ ]:
df.to_csv('ev_adoption_cleaned.csv', index=False)
print('Saved: ev_adoption_cleaned.csv')